In [ ]:
# Databricks notebook source
# tutorial_name: 05-Day8-Medallion-Gold-Task
# notebookName: 05-Day8-Medallion-Gold-Task
# COMMAND ----------

# Databricks notebook source
# COMMAND ----------

# DBTITLE 1,Paths (same as Days 5–8)
notepoint_rel = "hands-on/day-08/notebooks/05-Day8-Medallion-Gold-Task.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
SOURCE_JSON = BASE_PATH + "/json/2015-summary.json"
DAY8_ROOT = f"{BASE_PATH}day08-{STUDENT_ID}"
MED_ROOT = f"{DAY8_ROOT}/medallion"
BRONZE_PATH = f"{MED_ROOT}/bronze_flight_routes"
SILVER_PATH = f"{MED_ROOT}/silver_flight_routes"
GOLD_PATH = f"{MED_ROOT}/gold_by_destination"
METRICS_PATH = f"{DAY8_ROOT}/workflow_metrics/run_metrics"
# COMMAND ----------
print("DAY8_ROOT:", DAY8_ROOT)
print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("OK: P_BASIC")
except Exception as e:
    print("P_BASIC:", e)
try:
    spark.read.format("csv").option("header", True).load(SOURCE_CSV).limit(1).collect()
    print("OK: SOURCE_CSV")
except Exception as e:
    print("SOURCE_CSV:", e)
try:
    spark.read.format("json").load(SOURCE_JSON).limit(1).collect()
    print("OK: SOURCE_JSON (optional)")
except Exception as e:
    print("(optional) SOURCE_JSON:", type(e).__name__)
# COMMAND ----------


## Gold task (item 22)

**Purpose:** Aggregate Silver to **one row per `DEST_COUNTRY_NAME`** with **`total_flights`** = sum of **`count`**, add **`gold_built_ts`**.

**Requires:** Silver exists at **`SILVER_PATH`**.

**Also:** Appends one row to **`METRICS_PATH`** (pipeline metrics pattern) with `last_stage = gold`.


In [ ]:
from pyspark.sql import functions as F

df = spark.read.format("delta").load(SILVER_PATH)
out = df.groupBy("DEST_COUNTRY_NAME").agg(F.sum("count").alias("total_flights"))
out = out.withColumn("gold_built_ts", F.current_timestamp())
out.write.format("delta").mode("overwrite").save(GOLD_PATH)
print("Gold rows:", out.count(), "->", GOLD_PATH)
spark.read.format("delta").load(GOLD_PATH).orderBy(F.desc("total_flights")).show(15, truncate=False)
# COMMAND ----------


## Append metrics row (optional observability pattern)

In [ ]:
try:
    gold_n = spark.read.format("delta").load(GOLD_PATH).count()
except Exception:
    gold_n = -1
metrics = spark.sql(
    f"SELECT '{STUDENT_ID}' AS student_id, 'gold' AS last_stage, "
    f"current_timestamp() AS metric_ts, {int(gold_n)} AS gold_dim_rows"
)
metrics.write.format("delta").mode("append").save(METRICS_PATH)
spark.read.format("delta").load(METRICS_PATH).orderBy("metric_ts", ascending=False).show(5, truncate=False)
# COMMAND ----------
